# ncpu-computer

Train one shared local NCA rule on binary addition represented as a ternary tape. All experiment hyperparameters are defined below; targets are used only by the loss.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import torch
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "run":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ncpu_computer import (
    ExperimentConfig, GeometryConfig, ModelConfig, TapeLayout, TaskDataset,
    TrainingConfig, addition_task, encode_strings, evaluate, load_model,
    save_gif, train_seeds, validate_experiment,
)
from ncpu_computer.evaluation import format_results

# Task and tape geometry
OPERAND_BITS = 4
TAPE_SLOTS = 12
STRIDE = 2
BORDER_LEFT = 3
BORDER_RIGHT = 3
BORDER_TOP = 3
BORDER_BOTTOM = 3

# Local rule
CHANNELS = 5
HIDDEN_SIZE = 96
FIXED_KERNELS = ("identity", "sobel_x", "sobel_y")
FIXED_LAPLACIAN = False
LEARNABLE_KERNELS = 0
LEARNABLE_KERNEL_INIT = "laplacian"
GATE = "none"
GATE_BIAS = 1.0
FIRE_RATE = 0.95
PROGRAM_CHANNEL = 0
IO_CHANNEL = 1
PADDING = "zeros"
MAX_ABS_STATE = 10.0
RANDOM_KERNEL_SEED = 0

# Training
UPDATES = 2000
BATCH_SIZE = 64
FREE_STEPS = 30
SUPERVISION_STEPS = 60
LEARNING_RATE = 5e-3
FINAL_LEARNING_RATE = 1e-4
WARMUP_UPDATES = 0
WEIGHT_DECAY = 2e-5
GRAD_CLIP = 0.9
TERMINATOR_WEIGHT = 0.0
TAIL_WEIGHT = 0.0
TRAINING_SEED = 0
VALIDATION_EVERY = 50
CHECKPOINT_EVERY = 50
DEVICE = "auto"
SEEDS = (0,)

# Notebook actions
RUN_TRAINING = False
RESUME = False
RUN_TRAINING_EVALUATION = False
RUN_8BIT_EVALUATION = False
RUN_VISUALIZATION = False

# Evaluation and visualization
EVALUATION_BITS = 8
EVALUATION_TAPE_SLOTS = 20
EVALUATION_BATCH_SIZE = 256
EVALUATION_MAX_EXAMPLES = None
VISUALIZATION_INPUT = "1111B1"
VISUALIZATION_TARGET = "10000"
VISUALIZATION_STEPS = 200
GIF_DURATION_MS = 70
GIF_SCALE = 24
GIF_PATH = ROOT / "run" / "addition_evolution.gif"

geometry = GeometryConfig(
    tape_slots=TAPE_SLOTS,
    stride=STRIDE,
    border_left=BORDER_LEFT,
    border_right=BORDER_RIGHT,
    border_top=BORDER_TOP,
    border_bottom=BORDER_BOTTOM,
)
model_config = ModelConfig(
    channels=CHANNELS,
    hidden_size=HIDDEN_SIZE,
    fixed_kernels=FIXED_KERNELS,
    fixed_laplacian=FIXED_LAPLACIAN,
    learnable_kernels=LEARNABLE_KERNELS,
    learnable_kernel_init=LEARNABLE_KERNEL_INIT,
    gate=GATE,
    gate_bias=GATE_BIAS,
    fire_rate=FIRE_RATE,
    program_channel=PROGRAM_CHANNEL,
    io_channel=IO_CHANNEL,
    padding=PADDING,
    max_abs_state=MAX_ABS_STATE,
    random_kernel_seed=RANDOM_KERNEL_SEED,
)
training_config = TrainingConfig(
    updates=UPDATES,
    batch_size=BATCH_SIZE,
    free_steps=FREE_STEPS,
    supervision_steps=SUPERVISION_STEPS,
    learning_rate=LEARNING_RATE,
    final_learning_rate=FINAL_LEARNING_RATE,
    warmup_updates=WARMUP_UPDATES,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    terminator_weight=TERMINATOR_WEIGHT,
    tail_weight=TAIL_WEIGHT,
    seed=TRAINING_SEED,
    validation_every=VALIDATION_EVERY,
    checkpoint_every=CHECKPOINT_EVERY,
    device=DEVICE,
)
config = ExperimentConfig(geometry, model_config, training_config)
train_task = addition_task(OPERAND_BITS)
train_data = TaskDataset.from_task(train_task, geometry.tape_slots)
layout = TapeLayout(geometry)
checkpoint_dir = ROOT / "checkpoints"
checkpoint_path = checkpoint_dir / "best.pt"

print(validate_experiment(config, train_data))
print("\nTape layout:")
print(layout.schema())
print(f"\ninput example:  {train_data.input_strings[-1]}")
print(f"target example: {train_data.target_strings[-1]}")

In [ ]:
if RUN_TRAINING:
    seed_results = train_seeds(
        config, train_data, seeds=SEEDS, checkpoint_dir=checkpoint_dir,
        resume=RESUME,
    )
    seed_results

In [ ]:
if RUN_TRAINING_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    result = evaluate(
        model, trained_config.geometry, train_data,
        steps=trained_config.training.rollout_steps,
        step_start=trained_config.training.supervision_start,
        step_end=trained_config.training.supervision_end,
        batch_size=EVALUATION_BATCH_SIZE,
    )
    print(format_results([result]))

In [ ]:
if RUN_8BIT_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    evaluation_geometry = replace(
        trained_config.geometry, tape_slots=EVALUATION_TAPE_SLOTS
    )
    evaluation_data = TaskDataset.from_task(
        addition_task(EVALUATION_BITS), evaluation_geometry.tape_slots
    )
    result = evaluate(
        model, evaluation_geometry, evaluation_data,
        steps=trained_config.training.rollout_steps,
        step_start=trained_config.training.supervision_start,
        step_end=trained_config.training.supervision_end,
        batch_size=EVALUATION_BATCH_SIZE,
        max_examples=EVALUATION_MAX_EXAMPLES,
    )
    print(format_results([result]))

In [ ]:
if RUN_VISUALIZATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    visualization_layout = TapeLayout(trained_config.geometry)
    encoded = encode_strings(
        (VISUALIZATION_INPUT,), trained_config.geometry.tape_slots
    ).to(model.device)
    initial = model.initial_state(visualization_layout.render_tape(encoded))
    with torch.inference_mode():
        rollout = model(initial, VISUALIZATION_STEPS)[0].cpu()
    gif_path = save_gif(
        rollout, GIF_PATH, layout=visualization_layout, config=trained_config,
        input_symbols=VISUALIZATION_INPUT, target_symbols=VISUALIZATION_TARGET,
        output_mode=train_data.output_mode, duration_ms=GIF_DURATION_MS,
        scale=GIF_SCALE,
    )
    display(Image(filename=str(gif_path)))
    print(gif_path)